# 29 · Theory — Storage, Indexes & the Query Optimizer

The final theory piece: how the data physically lives on disk, what an index
*really* is, how the planner decides, and SQLite's unusual **type system**. This
is the mental model behind every performance decision.

In [ ]:
# ▶ Run this cell first. It loads JupySQL and connects to the SQLite database.
%load_ext sql
from sqlalchemy import create_engine
import os

# Works whether the notebook's working dir is the repo root or notebooks/
db_path = 'data/retail.db' if os.path.exists('data/retail.db') else '../data/retail.db'
engine = create_engine(f'sqlite:///{db_path}')

%config SqlMagic.autopandas = True      # results come back as pandas DataFrames
%config SqlMagic.displaycon = False
%config SqlMagic.feedback = 0
%config SqlMagic.displaylimit = 100

%sql engine
print('Connected to', db_path)

## How the database is stored
- A SQLite database is a **single file** divided into fixed-size **pages** (often
  4096 bytes).
- Each **table** and each **index** is stored as a **B-tree** of pages.
- Default tables are **rowid tables**: every row has a hidden 64-bit `rowid`, and
  the table B-tree is keyed by it. A `WITHOUT ROWID` table is keyed by its primary
  key instead (good for large text PKs).

See the physical layout of *this* database:

In [ ]:
%%sql
PRAGMA page_size;

In [ ]:
%%sql
PRAGMA page_count;

## What a B-tree buys you
A B-tree is **balanced** and keeps keys **sorted**, giving:
- **O(log n)** lookups by key (vs O(n) scanning every row),
- efficient **range scans** (`BETWEEN`, `>`, `ORDER BY`) because leaf pages are
  sorted,
- ordered traversal without a separate sort.

## What an index really is
An index is just **another B-tree** holding a *sorted copy* of the chosen
column(s) plus a pointer (the rowid) back to the full row. That's why:
- lookups/`ORDER BY` on indexed columns are fast,
- the **leftmost-prefix rule** exists (a `(a,b)` index is sorted by `a` then `b`,
  so it can't help a query filtering on `b` alone),
- a **covering index** (one that contains every column the query needs) avoids
  touching the table at all.

## The optimizer is cost-based
SQLite's planner estimates the cost of alternative plans and picks the cheapest.
It relies on **statistics** gathered by `ANALYZE` (stored in `sqlite_stat1`):
roughly, how many rows an index lookup will return (**selectivity**). A highly
*selective* filter (few matching rows) favors an index; a low-selectivity one
(matches most rows) favors a full scan.

Create an index, gather stats, and inspect them:

In [ ]:
%%sql
DROP INDEX IF EXISTS demo_idx_orders_cust;
CREATE INDEX demo_idx_orders_cust ON orders(customer_id);
ANALYZE;
SELECT * FROM sqlite_stat1 WHERE tbl = 'orders';

Each `sqlite_stat1` row reads roughly *"for this index, an average key matches N rows"* — the numbers the planner uses to choose.

## Two flavors of `EXPLAIN`
- **`EXPLAIN QUERY PLAN`** — the high-level strategy (`SCAN` vs `SEARCH ... USING
  INDEX`). This is what you read 99% of the time.
- **`EXPLAIN`** (no "QUERY PLAN") — the low-level **VM bytecode** SQLite compiles
  your query into. Rarely needed, but it shows SQL is *compiled*, not interpreted
  row-by-row.

In [ ]:
%%sql
EXPLAIN QUERY PLAN
SELECT * FROM orders WHERE customer_id = 1;

## Join algorithm
SQLite joins with a **nested-loop**: for each row of the outer table, it looks up
matching rows in the inner table. Without an index that's O(n × m); **with an
index on the inner join column** it becomes O(n × log m). The planner also
chooses the **join order** (which table is outer) based on cost. This is the
single biggest reason to index foreign-key columns you join on.

In [ ]:
%%sql
EXPLAIN QUERY PLAN
SELECT c.first_name, o.order_id
FROM customers c JOIN orders o ON o.customer_id = c.customer_id;

## Clean up the demo index

In [ ]:
%%sql
DROP INDEX IF EXISTS demo_idx_orders_cust;
SELECT 'cleaned up' AS status;

## SQLite's type system — storage classes & affinity
Unlike most databases, SQLite uses **dynamic typing**. A *value's* type is one of
five **storage classes**:

In [ ]:
%%sql
SELECT typeof(42) AS int_, typeof(3.14) AS real_, typeof('hi') AS text_, typeof(NULL) AS null_, typeof(x'01') AS blob_;

A *column's* declared type only sets a **type affinity** — a *preference* SQLite
uses to convert incoming values when it reasonably can. The five affinities are
`TEXT`, `NUMERIC`, `INTEGER`, `REAL`, `BLOB`, determined from the declared type
name. Watch what actually gets stored:

In [ ]:
%%sql
DROP TABLE IF EXISTS demo_affinity;
CREATE TABLE demo_affinity (
    i INTEGER,   -- INTEGER affinity
    t TEXT,      -- TEXT affinity
    r REAL,      -- REAL affinity
    b BLOB,      -- BLOB affinity (no conversion)
    n NUMERIC    -- NUMERIC affinity
);
INSERT INTO demo_affinity VALUES ('123', 123, '1.5', 9, '42');
SELECT typeof(i) AS i, typeof(t) AS t, typeof(r) AS r, typeof(b) AS b, typeof(n) AS n
FROM demo_affinity;

Notice: the text `'123'` inserted into the `INTEGER`-affinity column was
**converted to an integer**; the integer `123` inserted into the `TEXT` column
became **text**; and the `BLOB` column left `9` untouched. This flexible typing
is powerful but surprising — declare columns carefully and don't rely on the
database to reject a wrong-typed value the way a strict system would.

In [ ]:
%%sql
DROP TABLE IF EXISTS demo_affinity;
SELECT 'cleaned up' AS status;

## Practice

**✏️ Exercise 1.** Show the storage class SQLite assigns to the literals 100, 2.5, and 'sql' using typeof().

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
SELECT typeof(100) AS a, typeof(2.5) AS b, typeof('sql') AS c;

**✏️ Exercise 2.** Create an index on products(unit_price), run EXPLAIN QUERY PLAN for a price range query to confirm it's used, then drop the index.

Try it yourself in the practice cell, then run the solution to check.

In [ ]:
%%sql
-- Your turn! Replace the line below with your own query.
SELECT 'edit me, then run' AS your_answer;

<details><summary>💡 Show solution</summary>

Run the next cell to see one correct answer.</details>

In [ ]:
%%sql
DROP INDEX IF EXISTS demo_idx_price;
CREATE INDEX demo_idx_price ON products(unit_price);
EXPLAIN QUERY PLAN SELECT * FROM products WHERE unit_price BETWEEN 30 AND 60;

### ✅ Recap
Data lives in fixed-size pages as B-trees; an index is a sorted B-tree copy of
columns (hence leftmost-prefix and covering indexes); the planner is cost-based
and driven by `ANALYZE` statistics; joins are nested loops that love indexes on
the inner key; and SQLite's dynamic typing means columns have *affinity*, not
strict types.

## 🎓 That's the whole bootcamp — practice, advanced, and theory.
You now understand SQL from the syntax down to the storage engine. Go build.